## Classification of an M3-Antagonist model

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import joblib

# =========================
# SETTINGS
# =========================
FP_RADIUS = 2
FP_NBITS = 2048
FP_USE_CHIRALITY = True

# SCHALTER: ECFP4 vs FCFP4
FP_USE_FEATURES = False   # False=ECFP4, True=FCFP4

IN_CSV  = r"C:\path\to\database.csv"
OUT_CSV = r"C:\path\to\screen_M3_antagonism_ranked.csv"
MODEL_PATH = r"C:\path\to\model.joblib"

# Optional: Training fingerprints for Applicability Domain
# (z.B. als RDKit ExplicitBitVect Liste gespeichert)
TRAIN_FP_PATH = r"C:\path\to\train_fps.joblib"  # optional, else set None
USE_AD = True

# =========================
# HELPERS
# =========================
def mol_from_smiles(sm):
    if not isinstance(sm, str) or not sm.strip():
        return None
    m = Chem.MolFromSmiles(sm)
    return m

def morgan_fp(mol):
    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=FP_RADIUS,
        nBits=FP_NBITS,
        useChirality=FP_USE_CHIRALITY,
        useFeatures=FP_USE_FEATURES
    )

def fp_to_numpy(fp):
    arr = np.zeros((FP_NBITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def max_tanimoto_to_train(fp, train_fps):
    # train_fps: list of ExplicitBitVect
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    return float(np.max(sims)) if sims else np.nan

# =========================
# LOAD
# =========================
df = pd.read_csv(IN_CSV)
if "id" not in df.columns:
    df["id"] = np.arange(len(df))

model = joblib.load(MODEL_PATH)

train_fps = None
if USE_AD and TRAIN_FP_PATH:
    train_fps = joblib.load(TRAIN_FP_PATH)

# =========================
# BUILD FEATURES
# =========================
mols = []
fps = []
ok_mask = []

for sm in df["smiles"].astype(str).tolist():
    m = mol_from_smiles(sm)
    if m is None:
        mols.append(None); fps.append(None); ok_mask.append(False); continue
    fp = morgan_fp(m)
    mols.append(m); fps.append(fp); ok_mask.append(True)

df["ok"] = ok_mask

# Feature matrix for model:
# If your model was trained on FP-only, X = FP bits.
# If it was trained on FP+physchem and you saved a Pipeline that computes physchem internally,
# you must feed whatever that pipeline expects. Most robust is: Pipeline expects raw SMILES or mols.
#
# Here: assume model expects FP bits matrix.
X = np.vstack([fp_to_numpy(fp) for fp in fps if fp is not None])

# =========================
# PREDICT
# =========================
proba = model.predict_proba(X)[:, 1]  # class 1 = antagonist (assumption)
df.loc[df["ok"], "p_antagonist"] = proba

# =========================
# APPLICABILITY DOMAIN (optional)
# =========================
if train_fps is not None:
    max_sims = []
    for fp in fps:
        if fp is None:
            max_sims.append(np.nan)
        else:
            max_sims.append(max_tanimoto_to_train(fp, train_fps))
    df["max_sim_to_train"] = max_sims
    # Example flags
    df["in_domain"] = df["max_sim_to_train"] >= 0.3

# =========================
# RANK & EXPORT
# =========================
df_ranked = df[df["ok"]].sort_values("p_antagonist", ascending=False)
df_ranked.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV, "| n_ok:", df_ranked.shape[0])
